# **DONNEES ARGICOLES** 

## **1. Chargement et importation des librairies et du dataset**

In [1]:
import pandas as pd

excel_path = "recensement_donnee_agricole.xls"
xls = pd.ExcelFile(excel_path)


## **2. Définition des structures pour l'extraction**

In [2]:
# Liste globale qui contiendra les DataFrames nettoyés de chaque feuille
all_dfs = []

## **3. Boucle d'extraction et de transformation feuille par feuille**


In [3]:
# Parcourir chaque produit (feuille Excel)
for sheet_name in xls.sheet_names:
    df_raw = pd.read_excel(xls, sheet_name=sheet_name, header=None)

    # 1. Repérer la ligne d'en-tête (contient 'COMMUNES' ou 'COMMUNE')
    header_row_idx = None
    for idx, row in df_raw.iterrows():
        row_str = " ".join([str(v) for v in row.values if pd.notna(v)])
        if "COMMUNES" in row_str or "COMMUNE" in row_str:
            header_row_idx = idx
            break

    if header_row_idx is None:
        continue

    row_years = df_raw.iloc[header_row_idx].values
    row_metrics = df_raw.iloc[header_row_idx + 1].values

    # 2. Reconstruire la correspondance des colonnes (Année + Variable)
    cols_info = []
    current_year = None
    for i in range(len(row_years)):
        y = row_years[i]
        m = row_metrics[i]

        if pd.notna(y) and str(y).strip() not in [
            "N°",
            "COMMUNES",
            "COMMUNE",
            "PROD",
        ]:
            current_year = str(y).strip()

        if i == 0:
            cols_info.append(("META", "NO"))
        elif i == 1:
            cols_info.append(("META", "COMMUNE"))
        else:
            m_str = str(m).strip() if pd.notna(m) else ""
            if "SUP" in m_str.upper():
                metric = "superficie"
            elif "REND" in m_str.upper():
                metric = "rendement"
            elif "PROD" in m_str.upper():
                metric = "production"
            else:
                metric = f"unknown_{i}"
            cols_info.append((current_year, metric))

    # 3. Extraction des lignes de communes (exclut les totaux régionaux/nationaux)
    df_data = df_raw.iloc[header_row_idx + 2 :].copy()
    records = []

    for _, row in df_data.iterrows():
        num_val = row.values[0]
        commune_val = row.values[1]

        if pd.notna(commune_val) and pd.notna(num_val):
            try:
                # Vérifie que la ligne correspond à un numéro de commune valide (1 à 77)
                int(num_val)
                commune_name = str(commune_val).strip()

                for col_idx in range(2, len(cols_info)):
                    yr, met = cols_info[col_idx]
                    val = row.values[col_idx]
                    if yr and yr != "None" and "unknown" not in met:
                        records.append({
                            "produit": sheet_name,
                            "communes": commune_name,
                            "annee": yr,
                            "variable": met,
                            "valeur": val,
                        })
            except ValueError:
                continue

    if not records:
        continue

    # 4. Restructuration sous forme de tableau (Produit x Année x Commune)
    df_melted = pd.DataFrame(records)
    df_pivoted = df_melted.pivot_table(
        index=["produit", "annee", "communes"],
        columns="variable",
        values="valeur",
        aggfunc="first",
    ).reset_index()

    all_dfs.append(df_pivoted)

## **4. Concaténation, typage et réorganisation des colonnes**

In [4]:
# 5. Fusion globale de toutes les feuilles
df_complet = pd.concat(all_dfs, ignore_index=True)

# Nettoyage et conversion des types numériques
for col in ["production", "rendement", "superficie"]:
    if col in df_complet.columns:
        df_complet[col] = pd.to_numeric(df_complet[col], errors="coerce")

# Réorganisation des colonnes dans l'ordre désiré
df_complet = df_complet[
    ["produit", "annee", "communes", "superficie", "production", "rendement"]
]

print("Extraction et structuration complétées avec succès !")

Extraction et structuration complétées avec succès !


## **5. Contrôle du résultat et sauvegarde en CSV**

In [5]:
# Affichage des dimensions et des premières lignes
print("Nombre total d'observations :", len(df_complet))
print(df_complet.head(10))

# Sauvegarde dans un fichier CSV plat propre
df_complet.to_csv("donnees_agricoles_dataframe.csv", index=False)

Nombre total d'observations : 36574
variable produit      annee         communes  superficie  production  \
0           Maïs  1995-1996           ABOMEY      1317.0      1237.0   
1           Maïs  1995-1996    ABOMEY-CALAVI     21056.0     19519.0   
2           Maïs  1995-1996       ADJA-OUERE     29563.0     30317.0   
3           Maïs  1995-1996          ADJARRA      1705.0      2006.0   
4           Maïs  1995-1996         ADJOHOUN     11396.0     13518.0   
5           Maïs  1995-1996     AGBANGNIZOUN      3677.0      2692.0   
6           Maïs  1995-1996         AGUEGUES       575.0       756.0   
7           Maïs  1995-1996  AKPRO-MISSERETE      3922.0      4458.0   
8           Maïs  1995-1996           ALLADA     17895.0     15891.0   
9           Maïs  1995-1996         APLAHOUE      8926.0      7332.0   

variable    rendement  
0          939.255885  
1          927.004179  
2         1025.504854  
3         1176.539589  
4         1186.205686  
5          732.118575  
6  

### **AFFICHAGE**

In [6]:
data1 = pd.read_csv("donnees_agricoles_dataframe.csv")
data1.head(20)

,produit,annee,communes,superficie,production,rendement
0,Maïs,1995-1996,ABOMEY,1317.0,1237.0,939.255885
1,Maïs,1995-1996,ABOMEY-CALAVI,21056.0,19519.0,927.004179
2,Maïs,1995-1996,ADJA-OUERE,29563.0,30317.0,1025.504854
3,Maïs,1995-1996,ADJARRA,1705.0,2006.0,1176.539589
4,Maïs,1995-1996,ADJOHOUN,11396.0,13518.0,1186.205686
5,Maïs,1995-1996,AGBANGNIZOUN,3677.0,2692.0,732.118575
6,Maïs,1995-1996,AGUEGUES,575.0,756.0,1314.782609
7,Maïs,1995-1996,AKPRO-MISSERETE,3922.0,4458.0,1136.664967
8,Maïs,1995-1996,ALLADA,17895.0,15891.0,888.013412
9,Maïs,1995-1996,APLAHOUE,8926.0,7332.0,821.420569


## **6. Suppression des lignes des totaux departementaux**

In [7]:

# Suppression des lignes de totaux qui commencent par 'DEPART.' ou 'TOTAL' ou 'BENIN'
data1 = data1[
    ~data1["communes"].str.upper().str.startswith("DEPART.")
    & ~data1["communes"].str.upper().isin(["BENIN", "TOTAL"])
].copy()

print("Nombre de lignes avant filtrage :", len(data1))
print("Nombre de lignes après filtrage :", len(data1))
print("\nAperçu des communes restant dans le dataset :")
print(data1["communes"].unique()[:10])

Nombre de lignes avant filtrage : 36574
Nombre de lignes après filtrage : 36574

Aperçu des communes restant dans le dataset :
<ArrowStringArray>
[         'ABOMEY',   'ABOMEY-CALAVI',      'ADJA-OUERE',         'ADJARRA',
        'ADJOHOUN',    'AGBANGNIZOUN',        'AGUEGUES', 'AKPRO-MISSERETE',
          'ALLADA',        'APLAHOUE']
Length: 10, dtype: str


## 7. **Nettoyage des observations de la variable "communes"**

In [8]:
# Affichage du nombre de communes ainsi que leurs listes 
nb_communes = data1["communes"].unique()
print(f"Le nombre de communes pésente dans le jeu de donnée est {nb_communes}")
print(f"La liste de communes est {sorted(data1["communes"].unique())}")

Le nombre de communes pésente dans le jeu de donnée est <ArrowStringArray>
[         'ABOMEY',   'ABOMEY-CALAVI',      'ADJA-OUERE',         'ADJARRA',
        'ADJOHOUN',    'AGBANGNIZOUN',        'AGUEGUES', 'AKPRO-MISSERETE',
          'ALLADA',        'APLAHOUE',
 ...
       'Tanguiéta',       'Tchaourou',    'Toucountouna',        'Toviklin',
        'Za-Kpota',       'Zagnanado',      'Zogbodomey',   'ABOMEY CALAVI',
            'TORI',         'Athiémé']
Length: 137, dtype: str
La liste de communes est ['ABOMEY', 'ABOMEY CALAVI', 'ABOMEY-CALAVI', 'ADJA-OUERE', 'ADJARRA', 'ADJOHOUN', 'AFANGNI', 'AGBANGNIZOUN', 'AGUEGUES', 'AKPRO-MISSERETE', 'ALLADA', 'APLAHOUE', 'ATHIEME', 'AVRANKOU', 'Abomey', 'Adja-Ouèrè', 'Agbangnizoun', 'Aplahoué', 'Athiémé', 'BANIKOARA', 'BANTE', 'BASSILA', 'BEMBEREKE', 'BOHICON', 'BONOU', 'BOPA', 'BOUKOUMBE', 'Banikoara', 'Bantè', 'Bassila', 'Bembèrèkè', 'Bohicon', 'Bonou', 'Boukoumbé', 'COBLY', 'COME', 'COPARGO', 'COTONOU', 'COVE', 'Cobly', 'Copargo', 'Cov

In [9]:
import unicodedata

# 1. Suppression des espaces de début/fin et mise en majuscules
data1["communes"] = data1["communes"].str.strip().str.upper()

# 2. Suppression des accents
data1["communes"] = data1["communes"].apply(
    lambda x: "".join(
        c
        for c in unicodedata.normalize("NFD", x)
        if unicodedata.category(c) != "Mn"
    )
    if isinstance(x, str)
    else x
)

# 3. Remplacement des espaces internes et doubles tirets par des traits d'union simples
data1["communes"] = (
    data1["communes"]
    .str.replace(r"\s+", "-", regex=True)
    .str.replace(r"-+", "-", regex=True)
)

# 4. Correction des erreurs de frappe dans le nom des communes 
corrections = {
    "AFANGNI": "IFANGNI",
    "PEHOUCO": "PEHOUNCO",
    "PEHUNCO": "PEHOUNCO",
    "TANGUIETE": "TANGUIETA",
    "TORI": "TORI-BOSSITO",
}

data1["communes"] = data1["communes"].replace(corrections)
# 5. Vérification du nombre réel de communes uniques
nb_communes = data1["communes"].nunique()
print("Nombre de communes uniques après nettoyage :", nb_communes)

# 6. Affichage de la liste triée
print("\nListe des communes uniques :")
print(sorted(data1["communes"].unique()))

Nombre de communes uniques après nettoyage : 77

Liste des communes uniques :
['ABOMEY', 'ABOMEY-CALAVI', 'ADJA-OUERE', 'ADJARRA', 'ADJOHOUN', 'AGBANGNIZOUN', 'AGUEGUES', 'AKPRO-MISSERETE', 'ALLADA', 'APLAHOUE', 'ATHIEME', 'AVRANKOU', 'BANIKOARA', 'BANTE', 'BASSILA', 'BEMBEREKE', 'BOHICON', 'BONOU', 'BOPA', 'BOUKOUMBE', 'COBLY', 'COME', 'COPARGO', 'COTONOU', 'COVE', 'DANGBO', 'DASSA-ZOUME', 'DJAKOTOMEY', 'DJIDJA', 'DJOUGOU', 'DOGBO', 'GLAZOUE', 'GOGOUNOU', 'GRAND-POPO', 'HOUEYOGBE', 'IFANGNI', 'KALALE', 'KANDI', 'KARIMAMA', 'KEROU', 'KETOU', 'KLOUEKANME', 'KOUANDE', 'KPOMASSE', 'LALO', 'LOKOSSA', 'MALANVILLE', 'MATERI', "N'DALI", 'NATITINGOU', 'NIKKI', 'OUAKE', 'OUESSE', 'OUIDAH', 'OUINHI', 'PARAKOU', 'PEHOUNCO', 'PERERE', 'POBE', 'PORTO-NOVO', 'SAKETE', 'SAVALOU', 'SAVE', 'SEGBANA', 'SEME-PODJI', 'SINENDE', 'SO-AVA', 'TANGUIETA', 'TCHAOUROU', 'TOFFO', 'TORI-BOSSITO', 'TOUCOUNTOUNA', 'TOVIKLIN', 'ZA-KPOTA', 'ZAGNANADO', 'ZE', 'ZOGBODOMEY']


## **8. Nettoyage des données de la variables "annee"**

In [10]:
print(sorted(data1["annee"].unique()))

['1995-1996', '1996-1997', '1997-1998', '1998-1999', '1999-2000', '2000-2001', '2001-2002', '2002', '2002-2003', '2003', '2003-2004', '2004', '2004-2005', '2005', '2005-2006', '2006', '2006-2007', '2007', '2007-2008', '2008', '2008-2009', '2009', '2009-2010', '2010', '2010-2011', '2011', '2011-2012', '2012', '2012-2013', '2013', '2013-2014', '2014', '2014-2015', '2015', '2015-2016', '2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']


In [11]:
# Dictionnaire pour mapper les années simples vers le format de campagne agricole
corrections_annees = {
    "2002": "2002-2003",
    "2003": "2003-2004",
    "2004": "2004-2005",
    "2005": "2005-2006",
    "2006": "2006-2007",
    "2007": "2007-2008",
    "2008": "2008-2009",
    "2009": "2009-2010",
    "2010": "2010-2011",
    "2011": "2011-2012",
    "2012": "2012-2013",
    "2013": "2013-2014",
    "2014": "2014-2015",
    "2015": "2015-2016",
    "2016": "2016-2017",
}

# Application du dictionnaire sur la colonne annee
data1["annee"] = data1["annee"].replace(corrections_annees)

# Vérification des années uniques nettoyées
print("Nombre d'années uniques après nettoyage :", data1["annee"].nunique())
print("\nListe des années nettoyées :")
print(sorted(data1["annee"].unique()))

Nombre d'années uniques après nettoyage : 30

Liste des années nettoyées :
['1995-1996', '1996-1997', '1997-1998', '1998-1999', '1999-2000', '2000-2001', '2001-2002', '2002-2003', '2003-2004', '2004-2005', '2005-2006', '2006-2007', '2007-2008', '2008-2009', '2009-2010', '2010-2011', '2011-2012', '2012-2013', '2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']


## **9. Nettoyage des données de la variables "produit"**

In [12]:
print(sorted(data1["produit"].unique()))

['Anacarde', 'Ananas', 'Arachide', 'COTON', 'Carotte', 'Choux', 'Citulus', 'Concombre', 'Crincrin', 'Dohi', 'Fonio', 'Gboma', 'Gombo', 'Goussi', 'Haricot_vert', 'Igname', 'Laitue', 'Manioc', 'Maïs', 'Niébé', 'Oignon', 'Pasteque', 'Patate douce', 'Petit mil', 'Piment', "Poids d'angole", 'Pomme terre', 'Riz', 'Soja', 'Sorgho', 'Sésame', 'Taro', 'Tomate', 'Voandzou']


In [13]:
correction_produit = {'Haricot_vert':'Haricot vert','Maïs':'Mais', 'Niébé' :'Niebe', 'Sésame':'Sesame'
}
data1['produit'] = data1['produit'].str.replace(correction_produit)
data1['produit'] = data1['produit'].str.upper()

In [14]:
data1.shape

(36574, 6)

# **DONNEES METEOROLOGIQUES**

## **1. Importation des bibliothèques et configuration des listes de fichiers et mois**

In [15]:
import os
import pandas as pd

# Liste de vos fichiers météo
fichiers_meteo = [
    "donne_meteo_2012_2021.xlsx",
    "donne_meteo_2022.xlsx",
    "donne_meteo_2023.xlsx",
    "donne_meteo_2024.xlsx",
    "donne_meteo_2025.xlsx",
]

mois_liste = [
    "Janvier",
    "Février",
    "Mars",
    "Avril",
    "Mai",
    "Juin",
    "Juillet",
    "Août",
    "Septembre",
    "Octobre",
    "Novembre",
    "Décembre",
]

## **2. Définition de la fonction d'extraction météo**

In [16]:
def extraire_donnees_meteo(fichier_path):
    xls = pd.ExcelFile(fichier_path)
    records = []

    for sheet in xls.sheet_names:
        df_raw = pd.read_excel(xls, sheet_name=sheet, header=None)

        # Repérer la ligne d'en-tête (contient COMMUNES ou COMMUNE)
        header_idx = None
        for idx, row in df_raw.iterrows():
            row_str = " ".join([str(v) for v in row.values if pd.notna(v)])
            if "COMMUNE" in row_str.upper():
                header_idx = idx
                break

        if header_idx is None:
            continue

        row_header = df_raw.iloc[header_idx].values

        # Distinguer le fichier multi-années du fichier annuel
        is_multi = "ANNEE" in [
            str(v).upper() for v in row_header if pd.notna(v)
        ]

        # Extraire l'année par défaut à partir du nom de fichier si annuel
        annee_defaut = "".join(
            filter(str.isdigit, fichier_path.replace("2012_2021", ""))
        )

        # Parcourir les lignes de données
        for r_idx in range(header_idx + 2, len(df_raw)):
            row = df_raw.iloc[r_idx].values

            if is_multi:
                commune = row[0]
                annee_val = row[1]
                col_offset = 2
            else:
                num = row[0]
                commune = row[1]
                annee_val = annee_defaut
                col_offset = 2

                # Ne filtrer que les 77 communes (numéro valide de 1 à 77)
                try:
                    int(num)
                except (ValueError, TypeError):
                    continue

            if pd.isna(commune) or str(commune).strip() == "":
                continue

            # Extrait les valeurs pour chacun des 12 mois
            for m_idx, mois_nom in enumerate(mois_liste):
                c_haut = col_offset + (m_idx * 2)
                c_nj = col_offset + (m_idx * 2) + 1

                if c_nj < len(row):
                    haut_val = row[c_haut]
                    nj_val = row[c_nj]

                    records.append({
                        "communes": str(commune).strip(),
                        "annee": int(annee_val),
                        "mois": mois_nom,
                        "num_mois": m_idx + 1,
                        "Haut": haut_val,
                        "N/J": nj_val,
                    })

    return pd.DataFrame(records)

## **3. Extraction et fusion des fichiers météo**

In [17]:
# Extraction et fusion de tous les fichiers
dfs = [extraire_donnees_meteo(f) for f in fichiers_meteo if os.path.exists(f)]
df_meteo_final = pd.concat(dfs, ignore_index=True)

## **4. Nettoyage numérique des données météo**

In [18]:
# Nettoyage numérique
df_meteo_final["Haut"] = pd.to_numeric(df_meteo_final["Haut"], errors="coerce")
df_meteo_final["N/J"] = pd.to_numeric(df_meteo_final["N/J"], errors="coerce")

## **5. Affichage des résultats et sauvegarde CSV**

In [19]:
print("--- DataFrame extrait avec succès ---")
print("Nombre total d'enregistrements :", len(df_meteo_final))
print(df_meteo_final.head(12))

# Sauvegarde du fichier intermédiaire
df_meteo_final.to_csv("donnees_meteo_dataframe.csv", index=False)

--- DataFrame extrait avec succès ---
Nombre total d'enregistrements : 14496
     communes  annee       mois  num_mois    Haut   N/J
0   BEMBEREKE   2012    Janvier         1    0.00   0.0
1   BEMBEREKE   2012    Février         2   45.90   1.0
2   BEMBEREKE   2012       Mars         3    0.00   0.0
3   BEMBEREKE   2012      Avril         4   73.70   3.0
4   BEMBEREKE   2012        Mai         5  107.98   5.0
5   BEMBEREKE   2012       Juin         6  133.25   7.0
6   BEMBEREKE   2012    Juillet         7  316.77  14.0
7   BEMBEREKE   2012       Août         8  136.18  12.0
8   BEMBEREKE   2012  Septembre         9  337.24  16.0
9   BEMBEREKE   2012    Octobre        10   88.97   8.0
10  BEMBEREKE   2012   Novembre        11    0.00   0.0
11  BEMBEREKE   2012   Décembre        12    0.00   0.0


## **6. Affichage du dataset de la meteorologies** 

In [21]:
data2 = pd.read_csv("donnees_meteo_dataframe.csv")
data2.head()

,communes,annee,mois,num_mois,Haut,N/J
0,BEMBEREKE,2012,Janvier,1,0.00,0.0
1,BEMBEREKE,2012,Février,2,45.90,1.0
2,BEMBEREKE,2012,Mars,3,0.00,0.0
3,BEMBEREKE,2012,Avril,4,73.70,3.0
4,BEMBEREKE,2012,Mai,5,107.98,5.0


## **7. Nettoyage des données de la variable "communes"**

In [22]:

print(data2["communes"].unique())

<ArrowStringArray>
[         'BEMBEREKE',             'KALALE',             'N'DALI',
              'NIKKI',            'PARAKOU',             'PERERE',
            'SINENDE',          'TCHAOUROU',     'DEPART. BORGOU',
          'BANIKOARA',           'GOGOUNOU',              'KANDI',
           'KARIMAMA',         'MALANVILLE',            'SEGBANA',
    'DEPART. ALIBORI',          'BOUKOUMBE',              'COBLY',
              'KEROU',            'KOUANDE',             'MATERI',
         'NATITINGOU',            'PEHUNCO',          'TANGUIETA',
       'TOUCOUNTOUNA',    'DEPART. ATACORA',            'BASSILA',
            'COPARGO',            'DJOUGOU',              'OUAKE',
      'DEPART. DONGA',             'ABOMEY',       'AGBANGNIZOUN',
            'BOHICON',               'COVE',             'DJIDJA',
             'OUINHI',          'ZAGNANADO',           'ZA-KPOTA',
         'ZOGBODOMEY',        'DEPART. ZOU',              'BANTE',
        'DASSA-ZOUME',            'GLAZOUE'

In [23]:
# Suppression des lignes de totaux (DEPART.-..., BENIN, TOTAL)
data2 = data2[
    ~data2["communes"].str.contains("DEPART", case=False, na=False)
    & ~data2["communes"].isin(["BENIN", "TOTAL"])
].copy()

# Affichage de contrôle
print("Nombre de communes uniques après filtrage :", data2["communes"].nunique())
print("\nAperçu des communes :")
print(sorted(data2["communes"].unique()))

Nombre de communes uniques après filtrage : 77

Aperçu des communes :
['ABOMEY', 'ABOMEY-CALAVI', 'ADJA-OUERE', 'ADJARRA', 'ADJOHOUN', 'AGBANGNIZOUN', 'AGUEGUES', 'AKPRO-MISSERETE', 'ALLADA', 'APLAHOUE', 'ATHIEME', 'AVRANKOU', 'BANIKOARA', 'BANTE', 'BASSILA', 'BEMBEREKE', 'BOHICON', 'BONOU', 'BOPA', 'BOUKOUMBE', 'COBLY', 'COME', 'COPARGO', 'COTONOU', 'COVE', 'DANGBO', 'DASSA-ZOUME', 'DJAKOTOMEY', 'DJIDJA', 'DJOUGOU', 'DOGBO', 'GLAZOUE', 'GOGOUNOU', 'GRAND-POPO', 'HOUEYOGBE', 'IFANGNI', 'KALALE', 'KANDI', 'KARIMAMA', 'KEROU', 'KETOU', 'KLOUEKANME', 'KOUANDE', 'KPOMASSE', 'LALO', 'LOKOSSA', 'MALANVILLE', 'MATERI', "N'DALI", 'NATITINGOU', 'NIKKI', 'OUAKE', 'OUESSE', 'OUIDAH', 'OUINHI', 'PARAKOU', 'PEHUNCO', 'PERERE', 'POBE', 'PORTO-NOVO', 'SAKETE', 'SAVALOU', 'SAVE', 'SEGBANA', 'SEME-PODJI', 'SINENDE', 'SO-AVA', 'TANGUIETA', 'TCHAOUROU', 'TOFFO', 'TORI-BOSSITO', 'TOUCOUNTOUNA', 'TOVIKLIN', 'ZA-KPOTA', 'ZAGNANADO', 'ZE', 'ZOGBODOMEY']


In [24]:
import unicodedata

# 1. Suppression des espaces de début/fin et mise en majuscules
data2["communes"] = data2["communes"].str.strip().str.upper()

# 2. Suppression des accents
data2["communes"] = data2["communes"].apply(
    lambda x: "".join(
        c
        for c in unicodedata.normalize("NFD", x)
        if unicodedata.category(c) != "Mn"
    )
    if isinstance(x, str)
    else x
)

# 3. Remplacement des espaces internes et doubles tirets par des traits d'union simples
data2["communes"] = (
    data2["communes"]
    .str.replace(r"\s+", "-", regex=True)
    .str.replace(r"-+", "-", regex=True)
)

# 4. Correction des erreurs de frappe dans le nom des communes 
corrections = {
    "AFANGNI": "IFANGNI",
    "PEHOUCO": "PEHOUNCO",
    "PEHUNCO": "PEHOUNCO",
    "TANGUIETE": "TANGUIETA",
    "TORI": "TORI-BOSSITO",
}

data2["communes"] = data2["communes"].replace(corrections)
# 5. Vérification du nombre réel de communes uniques
nb_communes = data2["communes"].nunique()
print("Nombre de communes uniques après nettoyage :", nb_communes)

# 6. Affichage de la liste triée
print("\nListe des communes uniques :")
print(sorted(data2["communes"].unique()))

Nombre de communes uniques après nettoyage : 77

Liste des communes uniques :
['ABOMEY', 'ABOMEY-CALAVI', 'ADJA-OUERE', 'ADJARRA', 'ADJOHOUN', 'AGBANGNIZOUN', 'AGUEGUES', 'AKPRO-MISSERETE', 'ALLADA', 'APLAHOUE', 'ATHIEME', 'AVRANKOU', 'BANIKOARA', 'BANTE', 'BASSILA', 'BEMBEREKE', 'BOHICON', 'BONOU', 'BOPA', 'BOUKOUMBE', 'COBLY', 'COME', 'COPARGO', 'COTONOU', 'COVE', 'DANGBO', 'DASSA-ZOUME', 'DJAKOTOMEY', 'DJIDJA', 'DJOUGOU', 'DOGBO', 'GLAZOUE', 'GOGOUNOU', 'GRAND-POPO', 'HOUEYOGBE', 'IFANGNI', 'KALALE', 'KANDI', 'KARIMAMA', 'KEROU', 'KETOU', 'KLOUEKANME', 'KOUANDE', 'KPOMASSE', 'LALO', 'LOKOSSA', 'MALANVILLE', 'MATERI', "N'DALI", 'NATITINGOU', 'NIKKI', 'OUAKE', 'OUESSE', 'OUIDAH', 'OUINHI', 'PARAKOU', 'PEHOUNCO', 'PERERE', 'POBE', 'PORTO-NOVO', 'SAKETE', 'SAVALOU', 'SAVE', 'SEGBANA', 'SEME-PODJI', 'SINENDE', 'SO-AVA', 'TANGUIETA', 'TCHAOUROU', 'TOFFO', 'TORI-BOSSITO', 'TOUCOUNTOUNA', 'TOVIKLIN', 'ZA-KPOTA', 'ZAGNANADO', 'ZE', 'ZOGBODOMEY']


In [25]:
data2["annee"].dtype

dtype('int64')

## **8. Nettoyage des données de la variable "annee"**

In [26]:
# Convertit toute la colonne en chaîne de caractères
data2['annee'] = data2['annee'].astype(str)

In [27]:
# Dictionnaire pour mapper les années simples vers le format de campagne agricole
corrections_annees = {
    "2002": "2002-2003",
    "2003": "2003-2004",
    "2004": "2004-2005",
    "2005": "2005-2006",
    "2006": "2006-2007",
    "2007": "2007-2008",
    "2008": "2008-2009",
    "2009": "2009-2010",
    "2010": "2010-2011",
    "2011": "2011-2012",
    "2012": "2012-2013",
    "2013": "2013-2014",
    "2014": "2014-2015",
    "2015": "2015-2016",
    "2016": "2016-2017",
    "2017": "2017-2018",
    "2018": "2018-2019",
    "2019": "2019-2020",
    "2020": "2020-2021",
    "2021": "2021-2022",
    "2022": "2022-2023",
    "2023": "2023-2024",
    "2024": "2024-2025",
    
}

# Application du dictionnaire sur la colonne annee
data2["annee"] = data2["annee"].replace(corrections_annees)

# Vérification des années uniques nettoyées
print("Nombre d'années uniques après nettoyage :", data2["annee"].nunique())
print("\nListe des années nettoyées :")
print(sorted(data2["annee"].unique()))

Nombre d'années uniques après nettoyage : 14

Liste des années nettoyées :
['2012-2013', '2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025']


## **9. Agrégation des données météo par commune et par année**

In [28]:
# Agrégation par commune et par année (suppression de la dimension mensuelle)
data2 = data2.groupby(["communes", "annee"]).agg(
    Haut_cumul=("Haut", "sum"),
    NJ_cumul=("N/J", "sum"),
    Haut_moyenne=("Haut", "mean")
).reset_index()

# Arrondi à 2 décimales pour garder des chiffres propres
data2 = data2.round(2)

## **10. Vérification du dataset météo annuel**

In [29]:
print("--- DataFrame météo agrégé par année ---")
print("Nombre de lignes (commune x année) :", len(data2))
print("\nAperçu des données :")
data2.head()

--- DataFrame météo agrégé par année ---
Nombre de lignes (commune x année) : 1078

Aperçu des données :


,communes,annee,Haut_cumul,NJ_cumul,Haut_moyenne
0,ABOMEY,2012-2013,997.10,62.00,83.09
1,ABOMEY,2013-2014,1002.98,42.00,83.58
2,ABOMEY,2014-2015,1001.07,49.65,83.42
3,ABOMEY,2015-2016,513.70,24.00,42.81
4,ABOMEY,2016-2017,807.10,35.00,67.26


## **11.FUSION DES DEUX DATASETS**

### **Fusion des données (Inner Join ou Left Join)**

In [30]:
# Fusion des datasets sur les clés 'communes' et 'annee'
data = pd.merge(
    data1,                 # ton dataset agricole nettoyé
    data2,                 # ton dataset agricole nettoyé
    on=["communes", "annee"],
    how="inner"             # 'left' pour conserver toutes les données agricoles
)

# Vérification du résultat
print("--- Résultat de la fusion ---")
print("Dimension du dataset final :", data.shape)
print("\nAperçu des premières lignes :")
print(data.head())

--- Résultat de la fusion ---
Dimension du dataset final : (20721, 9)

Aperçu des premières lignes :
  produit      annee       communes    superficie    production    rendement  \
0    MAIS  2012-2013         ABOMEY   2205.000000   2168.400000   983.401361   
1    MAIS  2012-2013  ABOMEY-CALAVI   9916.000000  14314.850000  1443.611335   
2    MAIS  2012-2013     ADJA-OUERE  12461.707252  13904.427053  1115.772243   
3    MAIS  2012-2013        ADJARRA   1324.033150   1520.088189  1148.074117   
4    MAIS  2012-2013       ADJOHOUN  35538.543501  49692.697250  1398.276135   

   Haut_cumul  NJ_cumul  Haut_moyenne  
0      997.10      62.0         83.09  
1     1186.46      35.0         98.87  
2     1100.96      47.0         91.75  
3     1308.30      69.0        109.02  
4     1344.61      66.0        112.05  


### **Contrôle des valeurs manquantes après fusion**

In [31]:
# Vérification des valeurs nulles apportées par la météo
météo_manquante = data["Haut_cumul"].isna().sum()
print(f"Nombre de lignes agricoles sans correspondance météo : {météo_manquante}")

if météo_manquante > 0:
    print("\nExemple de communes/années sans météo :")
    print(data[data["Haut_cumul"].isna()][["communes", "annee"]].drop_duplicates().head(10))

Nombre de lignes agricoles sans correspondance météo : 0


### **Sauvegarde du jeu de données final pour la modélisation**

In [32]:
# Sauvegarde du dataset complet nettoyé et fusionné
data.to_csv("donnees_nettoyees_benin.csv", index=False)
print("Fichier final 'donnees_nettoyees_benin.csv' enregistré avec succès !")

Fichier final 'donnees_nettoyees_benin.csv' enregistré avec succès !


### **Recalcul mathématiques**

In [33]:
# 1. Recalcul du rendement manquant (si superficie et production existent)
mask_rend = data["rendement"].isna() & data["superficie"].notna() & data["production"].notna()
data.loc[mask_rend, "rendement"] = data.loc[mask_rend, "production"] / data.loc[mask_rend, "superficie"]

# 2. Recalcul de la production manquante (si superficie et rendement existent)
mask_prod = data["production"].isna() & data["superficie"].notna() & data["rendement"].notna()
data.loc[mask_prod, "production"] = data.loc[mask_prod, "superficie"] * data.loc[mask_prod, "rendement"]

# 3. Recalcul de la superficie manquante (si production et rendement existent)
mask_sup = data["superficie"].isna() & data["production"].notna() & data["rendement"].notna()
data.loc[mask_sup, "superficie"] = data.loc[mask_sup, "production"] / data.loc[mask_sup, "rendement"]

print("Valeurs manquantes après recalcul mathématique :")
print(data[["superficie", "production", "rendement"]].isna().sum())

Valeurs manquantes après recalcul mathématique :
superficie      50
production      12
rendement     1910
dtype: int64


###  **Imputation des valeurs manquantes par la médiane**

In [34]:
# Imputation par la médiane au niveau (produit, commune)
data["superficie"] = data.groupby(["produit", "communes"])["superficie"].transform(lambda x: x.fillna(x.median()))
data["rendement"] = data.groupby(["produit", "communes"])["rendement"].transform(lambda x: x.fillna(x.median()))

# Recalcul de la production à partir des valeurs imputées
data["production"] = data["production"].fillna(data["superficie"] * data["rendement"])

# Imputation de sécurité par la médiane globale du produit (si la commune n'a aucun historique pour ce produit)
data["rendement"] = data.groupby("produit")["rendement"].transform(lambda x: x.fillna(x.median()))
data["superficie"] = data.groupby("produit")["superficie"].transform(lambda x: x.fillna(x.median()))
data["production"] = data["production"].fillna(data["superficie"] * data["rendement"])

In [35]:
data.to_csv("donnees_nettoyees_benin.csv", index=False)

In [36]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 20721 entries, 0 to 20720
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   produit       20721 non-null  str    
 1   annee         20721 non-null  str    
 2   communes      20721 non-null  str    
 3   superficie    20721 non-null  float64
 4   production    20721 non-null  float64
 5   rendement     20721 non-null  float64
 6   Haut_cumul    20721 non-null  float64
 7   NJ_cumul      20721 non-null  float64
 8   Haut_moyenne  20721 non-null  float64
dtypes: float64(6), str(3)
memory usage: 1.9 MB
